In [1]:
!pip install -q mlxtend

In [5]:
import pandas as pd

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
import logging
import warnings
import os

# reduz logs gerais
logging.getLogger().setLevel(logging.CRITICAL)

# reduz logs de bibliotecas comuns
for nome in ["matplotlib", "PIL", "numexpr", "tensorflow", "absl"]:
    logging.getLogger(nome).setLevel(logging.CRITICAL)

# ignora warnings
warnings.filterwarnings("ignore")

# TensorFlow, se estiver no notebook
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

In [8]:
dados = pd.DataFrame({
    "Aluno": ["A1", "A2", "A3", "A4", "A5", "A6", "A7", "A8", "A9", "A10"],
    "Frequencia": ["Alta", "Alta", "Média", "Baixa", "Baixa", "Alta", "Média", "Baixa", "Alta", "Média"],
    "HorasEstudo": ["Muitas", "Moderadas", "Moderadas", "Poucas", "Poucas", "Muitas", "Poucas", "Moderadas", "Muitas", "Moderadas"],
    "Internet": ["Boa", "Boa", "Regular", "Ruim", "Ruim", "Boa", "Regular", "Regular", "Boa", "Boa"],
    "Resultado": ["Aprovado", "Aprovado", "Aprovado", "Reprovado", "Reprovado", "Aprovado", "Reprovado", "Reprovado", "Aprovado", "Aprovado"]
})

dados

,Aluno,Frequencia,HorasEstudo,Internet,Resultado
0,A1,Alta,Muitas,Boa,Aprovado
1,A2,Alta,Moderadas,Boa,Aprovado
2,A3,Média,Moderadas,Regular,Aprovado
3,A4,Baixa,Poucas,Ruim,Reprovado
4,A5,Baixa,Poucas,Ruim,Reprovado
5,A6,Alta,Muitas,Boa,Aprovado
6,A7,Média,Poucas,Regular,Reprovado
7,A8,Baixa,Moderadas,Regular,Reprovado
8,A9,Alta,Muitas,Boa,Aprovado
9,A10,Média,Moderadas,Boa,Aprovado


In [9]:
transacoes = []

for _, linha in dados.iterrows():
    transacao = [
        f"Frequencia={linha['Frequencia']}",
        f"HorasEstudo={linha['HorasEstudo']}",
        f"Internet={linha['Internet']}",
        f"Resultado={linha['Resultado']}"
    ]
    transacoes.append(transacao)

transacoes

[['Frequencia=Alta',
  'HorasEstudo=Muitas',
  'Internet=Boa',
  'Resultado=Aprovado'],
 ['Frequencia=Alta',
  'HorasEstudo=Moderadas',
  'Internet=Boa',
  'Resultado=Aprovado'],
 ['Frequencia=Média',
  'HorasEstudo=Moderadas',
  'Internet=Regular',
  'Resultado=Aprovado'],
 ['Frequencia=Baixa',
  'HorasEstudo=Poucas',
  'Internet=Ruim',
  'Resultado=Reprovado'],
 ['Frequencia=Baixa',
  'HorasEstudo=Poucas',
  'Internet=Ruim',
  'Resultado=Reprovado'],
 ['Frequencia=Alta',
  'HorasEstudo=Muitas',
  'Internet=Boa',
  'Resultado=Aprovado'],
 ['Frequencia=Média',
  'HorasEstudo=Poucas',
  'Internet=Regular',
  'Resultado=Reprovado'],
 ['Frequencia=Baixa',
  'HorasEstudo=Moderadas',
  'Internet=Regular',
  'Resultado=Reprovado'],
 ['Frequencia=Alta',
  'HorasEstudo=Muitas',
  'Internet=Boa',
  'Resultado=Aprovado'],
 ['Frequencia=Média',
  'HorasEstudo=Moderadas',
  'Internet=Boa',
  'Resultado=Aprovado']]

In [10]:
te = TransactionEncoder()

matriz = te.fit(transacoes).transform(transacoes)

dados_transformados = pd.DataFrame(matriz, columns=te.columns_)

dados_transformados


,Frequencia=Alta,Frequencia=Baixa,Frequencia=Média,HorasEstudo=Moderadas,HorasEstudo=Muitas,HorasEstudo=Poucas,Internet=Boa,Internet=Regular,Internet=Ruim,Resultado=Aprovado,Resultado=Reprovado
0,True,False,False,False,True,False,True,False,False,True,False
1,True,False,False,True,False,False,True,False,False,True,False
2,False,False,True,True,False,False,False,True,False,True,False
3,False,True,False,False,False,True,False,False,True,False,True
4,False,True,False,False,False,True,False,False,True,False,True
5,True,False,False,False,True,False,True,False,False,True,False
6,False,False,True,False,False,True,False,True,False,False,True
7,False,True,False,True,False,False,False,True,False,False,True
8,True,False,False,False,True,False,True,False,False,True,False
9,False,False,True,True,False,False,True,False,False,True,False


In [11]:
itemsets_frequentes = apriori(
    dados_transformados,
    min_support=0.3,
    use_colnames=True
)

itemsets_frequentes.sort_values(by="support", ascending=False)

,support,itemsets
8,0.6,(Resultado=Aprovado)
6,0.5,(Internet=Boa)
18,0.5,"(Resultado=Aprovado, Internet=Boa)"
0,0.4,(Frequencia=Alta)
9,0.4,(Resultado=Reprovado)
11,0.4,"(Frequencia=Alta, Internet=Boa)"
3,0.4,(HorasEstudo=Moderadas)
21,0.4,"(Frequencia=Alta, Resultado=Aprovado, Internet..."
12,0.4,"(Frequencia=Alta, Resultado=Aprovado)"
7,0.3,(Internet=Regular)


In [12]:
regras = association_rules(
    itemsets_frequentes,
    metric="confidence",
    min_threshold=0.7
)

regras_resumidas = regras[
    ["antecedents", "consequents", "support", "confidence", "lift"]
]

regras_resumidas.sort_values(by="confidence", ascending=False)

,antecedents,consequents,support,confidence,lift
1,(HorasEstudo=Muitas),(Frequencia=Alta),0.3,1.000000,2.500000
2,(Frequencia=Alta),(Internet=Boa),0.4,1.000000,2.000000
4,(Frequencia=Alta),(Resultado=Aprovado),0.4,1.000000,1.666667
9,(HorasEstudo=Muitas),(Resultado=Aprovado),0.3,1.000000,1.666667
5,(Frequencia=Baixa),(Resultado=Reprovado),0.3,1.000000,2.500000
8,(HorasEstudo=Muitas),(Internet=Boa),0.3,1.000000,2.000000
18,(HorasEstudo=Muitas),"(Frequencia=Alta, Internet=Boa)",0.3,1.000000,2.500000
16,"(HorasEstudo=Muitas, Internet=Boa)",(Frequencia=Alta),0.3,1.000000,2.500000
14,"(Frequencia=Alta, HorasEstudo=Muitas)",(Internet=Boa),0.3,1.000000,2.000000
13,(Internet=Boa),(Resultado=Aprovado),0.5,1.000000,1.666667


In [13]:
def formatar_conjunto(conjunto):
    return ", ".join(list(conjunto))

regras_resumidas = regras_resumidas.copy()

regras_resumidas["Regra"] = regras_resumidas.apply(
    lambda linha: f"{formatar_conjunto(linha['antecedents'])} → {formatar_conjunto(linha['consequents'])}",
    axis=1
)

regras_resumidas[["Regra", "support", "confidence", "lift"]].sort_values(
    by="confidence",
    ascending=False
)

,Regra,support,confidence,lift
1,HorasEstudo=Muitas → Frequencia=Alta,0.3,1.000000,2.500000
2,Frequencia=Alta → Internet=Boa,0.4,1.000000,2.000000
4,Frequencia=Alta → Resultado=Aprovado,0.4,1.000000,1.666667
9,HorasEstudo=Muitas → Resultado=Aprovado,0.3,1.000000,1.666667
5,Frequencia=Baixa → Resultado=Reprovado,0.3,1.000000,2.500000
8,HorasEstudo=Muitas → Internet=Boa,0.3,1.000000,2.000000
18,"HorasEstudo=Muitas → Frequencia=Alta, Internet...",0.3,1.000000,2.500000
16,"HorasEstudo=Muitas, Internet=Boa → Frequencia=...",0.3,1.000000,2.500000
14,"Frequencia=Alta, HorasEstudo=Muitas → Internet...",0.3,1.000000,2.000000
13,Internet=Boa → Resultado=Aprovado,0.5,1.000000,1.666667


In [14]:
regras_aprovacao = regras_resumidas[
    regras_resumidas["consequents"].apply(lambda x: "Resultado=Aprovado" in x)
]

regras_aprovacao[["Regra", "support", "confidence", "lift"]]

,Regra,support,confidence,lift
4,Frequencia=Alta → Resultado=Aprovado,0.4,1.00,1.666667
7,HorasEstudo=Moderadas → Resultado=Aprovado,0.3,0.75,1.250000
9,HorasEstudo=Muitas → Resultado=Aprovado,0.3,1.00,1.666667
13,Internet=Boa → Resultado=Aprovado,0.5,1.00,1.666667
20,"Frequencia=Alta, HorasEstudo=Muitas → Resultad...",0.3,1.00,1.666667
22,"Frequencia=Alta → Resultado=Aprovado, HorasEst...",0.3,0.75,2.500000
23,"HorasEstudo=Muitas → Frequencia=Alta, Resultad...",0.3,1.00,2.500000
25,"Frequencia=Alta, Internet=Boa → Resultado=Apro...",0.4,1.00,1.666667
27,"Frequencia=Alta → Resultado=Aprovado, Internet...",0.4,1.00,2.000000
28,"Internet=Boa → Frequencia=Alta, Resultado=Apro...",0.4,0.80,2.000000


In [12]:
regras_reprovacao = regras_resumidas[
    regras_resumidas["consequents"].apply(lambda x: "Resultado=Reprovado" in x)
]

regras_reprovacao[["Regra", "support", "confidence", "lift"]]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,Regra,support,confidence,lift
6,Frequencia=Baixa → Resultado=Reprovado,0.3,1.0,2.5
10,HorasEstudo=Poucas → Resultado=Reprovado,0.3,1.0,2.5


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
